In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_6.py ──
"""
Shared infrastructure for MLFP03 Exercise 6 — Interpretability and Fairness.

Contains: Singapore credit scoring data load, LightGBM model training
(via kailash-ml TrainingPipeline),
TreeSHAP explainer setup, output directory, and common helper utilities.

Technique-specific code (permutation importance loops, LIME wrappers,
fairness audit reports) lives in the per-technique files under
`modules/mlfp03/solutions/ex_6/`.

Import pattern (solutions and local both):

"""

import asyncio
import os
import pickle
from pathlib import Path
from typing import Any

import warnings

import numpy as np
import polars as pl

import shap
from sklearn.metrics import roc_auc_score

from kailash_ml.interop import to_sklearn_input



# ════════════════════════════════════════════════════════════════════════
# PATHS / CONSTANTS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp03_ex6_interpretability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Singapore credit scoring: this synthetic dataset simulates a retail-bank
# default prediction task used throughout MLFP02/MLFP03. Banks are expected
# to be able to explain credit decisions (e.g. under the MAS FEAT
# principles, which are non-binding guidance), which is why Exercise 6
# explains this model.
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"
RANDOM_SEED = 42

# Columns that MUST NOT be model inputs (Lesson 3.1 leakage rule):
#   customer_id              — a row identifier, not a property of the applicant
#   future_default_indicator — recorded AFTER the loan outcome is known; it
#                              agrees with ``default`` on ~99% of rows, so a
#                              model that sees it "predicts" default by
#                              reading the answer (Exercise 4 screens for it).
CREDIT_NON_FEATURE_COLUMNS: tuple[str, ...] = ("customer_id", "future_default_indicator")

# Protected attributes audited in 05_fairness_audit.py. These are the
# REAL column names in sg_credit_scoring.parquet (race, not "ethnicity").
# Categorical ones are ordinal-encoded by PreprocessingPipeline and are
# decoded back to labels with ``decode_group``; age is banded.
PROTECTED_ATTRIBUTES: list[str] = ["race", "gender", "age"]
AGE_BANDS: list[tuple[str, int, int]] = [
    ("21-34", 21, 34),
    ("35-49", 35, 49),
    ("50-64", 50, 64),
    ("65+", 65, 200),
]


# ════════════════════════════════════════════════════════════════════════
# DATA LOAD + MODEL TRAIN
# ════════════════════════════════════════════════════════════════════════

# Populated on first call so every technique file sees the same split.
_CACHE: dict[str, Any] = {}


def load_credit_scoring() -> dict[str, Any]:
    """Load the Singapore credit scoring dataset and run the M3 preprocessing
    pipeline. Returns a dict with X_train, y_train, X_test, y_test, feature_names.

    The return value is cached so repeated calls from different technique
    files re-use the same split (essential for interpretability comparisons).
    """
    if _CACHE:
        return _CACHE

    loader = MLFPDataLoader()
    credit: pl.DataFrame = loader.load(DATASET_MODULE, DATASET_FILE).drop(
        CREDIT_NON_FEATURE_COLUMNS
    )

    # Split FIRST, then fit imputation/encoding on the training rows only:
    # PreprocessingPipeline.setup() on the whole frame would fit them on the
    # test rows too (it splits only after fitting).
    result = split_then_preprocess(
        credit,
        target=TARGET_COLUMN,
        test_size=0.2,
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_columns = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    feature_names: list[str] = col_info["feature_columns"]

    _CACHE.update(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        feature_names=feature_names,
        ordinal_mappings=result.transformers["ordinal_mappings"],
    )
    return _CACHE


# The model is trained through kailash-ml's TrainingPipeline (fit + holdout
# evaluation + registry entry in one call) and loaded back from the
# registry, so SHAP explains exactly the registered artefact. Class
# weighting (scale_pos_weight) is kept from the original exercise design:
# it raises recall on the 12% default class at the cost of inflated
# probabilities (Exercise 5) — 05_fairness_audit.py discusses the effect.
MODEL_NAME = "credit_default_ex6"
_DB_ABS_PATH = (OUTPUT_DIR / "ex6_models.db").resolve()
DB_URL: str = os.environ.get("MLFP03_EX6_DB_URL", f"sqlite:///{_DB_ABS_PATH.as_posix()}")


async def _train_via_pipeline(
    X_train: np.ndarray, y_train: np.ndarray, feature_names: list[str]
) -> Any:
    from kailash.db import ConnectionManager
    from kailash_ml import ModelRegistry, TrainingPipeline
    from kailash_ml.engines.training_pipeline import EvalSpec, ModelSpec
    from kailash_ml.types import FeatureField, FeatureSchema

    frame = pl.DataFrame(X_train, schema=feature_names, orient="row").with_columns(
        pl.Series(TARGET_COLUMN, y_train),
        pl.int_range(0, len(y_train), dtype=pl.Int64).alias("row_id"),
    )
    schema = FeatureSchema(
        name="ex6_credit_input",
        features=[FeatureField(name=f, dtype="float64") for f in feature_names],
        entity_id_column="row_id",
    )
    conn = ConnectionManager(DB_URL)
    await conn.initialize()
    try:
        registry = ModelRegistry(conn)
        pipeline = TrainingPipeline(feature_store=None, registry=registry)
        result = await pipeline.train(
            data=frame,
            schema=schema,
            model_spec=ModelSpec(
                model_class="lightgbm.LGBMClassifier",
                framework="lightgbm",
                hyperparameters={
                    "n_estimators": 500,
                    "learning_rate": 0.1,
                    "max_depth": 6,
                    "scale_pos_weight": float((1 - y_train.mean()) / y_train.mean()),
                    "random_state": RANDOM_SEED,
                    "verbose": -1,
                },
            ),
            eval_spec=EvalSpec(metrics=["auc"], split_strategy="holdout", test_size=0.2),
            experiment_name=MODEL_NAME,
        )
        if result.model_version is None:
            raise RuntimeError("TrainingPipeline did not register the model")
        version = int(result.model_version.version)
        # Unpickling executes code: only load artefacts you trained yourself.
        return pickle.loads(await registry.load_artifact(MODEL_NAME, version))
    finally:
        await conn.close()


def train_credit_model() -> dict[str, Any]:
    """Train the LightGBM credit default model via TrainingPipeline. Cached per-process.

    Returns a dict with model, y_proba, y_pred, auc, and all data from
    `load_credit_scoring()`.
    """
    if "model" in _CACHE:
        return _CACHE

    data = load_credit_scoring()
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]

    model = asyncio.run(_train_via_pipeline(X_train, y_train, data["feature_names"]))

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_proba)

    _CACHE.update(model=model, y_proba=y_proba, y_pred=y_pred, auc=auc)
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# SHAP EXPLAINER
# ════════════════════════════════════════════════════════════════════════


def build_shap_explainer() -> dict[str, Any]:
    """Construct the TreeSHAP explainer and compute SHAP values for X_test.

    Returns the full bundle: model, data, explainer, shap_vals, expected_value.
    """
    if "shap_vals" in _CACHE:
        return _CACHE

    bundle = train_credit_model()
    explainer = shap.TreeExplainer(bundle["model"])
    # shap emits a UserWarning announcing the binary-LightGBM output-shape
    # change ("...has changed to a list of ndarray"). The isinstance branch
    # below already handles both shapes, so the notice is informational only —
    # filtered narrowly, by exact message prefix, at the one call site.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"LightGBM binary classifier with TreeExplainer shap values output has changed.*",
            category=UserWarning,
        )
        shap_values = explainer.shap_values(bundle["X_test"])

    # TreeSHAP for binary classifiers may return [class_0, class_1]
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    expected_value = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, list)
        else explainer.expected_value
    )

    _CACHE.update(
        explainer=explainer,
        shap_vals=shap_vals,
        expected_value=expected_value,
    )
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# REUSABLE UTILITIES
# ════════════════════════════════════════════════════════════════════════


def rank_features_by_mean_abs_shap(
    shap_vals: np.ndarray, feature_names: list[str]
) -> list[tuple[str, float]]:
    """Return [(feature, mean_abs_shap), ...] sorted descending."""
    mean_abs = np.abs(shap_vals).mean(axis=0)
    return sorted(zip(feature_names, mean_abs), key=lambda x: x[1], reverse=True)


def feature_index(feature_names: list[str], name: str) -> int:
    """Lookup a feature column index by name, raising a clear error."""
    if name not in feature_names:
        raise KeyError(
            f"Feature '{name}' not found. Available: {feature_names[:10]}..."
        )
    return feature_names.index(name)


def decode_group(
    X: np.ndarray,
    feature_names: list[str],
    attribute: str,
    ordinal_mappings: dict[str, dict[str, int]],
) -> np.ndarray:
    """Return a string label per row for a protected attribute.

    Categorical attributes are mapped back from their ordinal codes using
    the PreprocessingPipeline's fitted ``ordinal_mappings``; ``age`` is
    binned into ``AGE_BANDS``.
    """
    values = X[:, feature_index(feature_names, attribute)]
    if attribute == "age":
        labels = np.full(values.shape[0], "unknown", dtype=object)
        for name, lo, hi in AGE_BANDS:
            labels[(values >= lo) & (values <= hi)] = name
        return labels
    if attribute not in ordinal_mappings:
        raise KeyError(f"No ordinal mapping for '{attribute}' — is it categorical?")
    code_to_label = {code: label for label, code in ordinal_mappings[attribute].items()}
    return np.array([code_to_label.get(int(v), "unknown") for v in values], dtype=object)


def print_section(title: str, char: str = "=") -> None:
    """Print a standardised section banner."""
    line = char * 70
    print(f"\n{line}")
    print(f"  {title}")
    print(line)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 6.5: Fairness Audit (Disparate Impact, Equalized Odds,
#                         Calibration Parity, Impossibility Theorem)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Measure the disparate impact ratio (the "four-fifths rule")
#   - Measure equalized odds (TPR and FPR parity across groups)
#   - Measure calibration parity (are probabilities equally reliable?)
#   - Work the Chouldechova/Kleinberg impossibility theorem with THIS
#     model's real base rates
#   - Produce a fairness audit report for race, gender and age
#
# PREREQUISITES:
#   - 01_shap_global.py (same model, same SHAP bundle)
#
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — why fairness has MULTIPLE incompatible definitions
#   2. Build — decode protected groups + per-group rate machinery
#   3. Train — no training; AUDIT the trained credit model
#   4. Visualise — per-group tables, charts, impossibility worked example
#   5. Apply — fairness disclosure for an (illustrative) retail bank
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from dotenv import load_dotenv

from kailash_ml import diagnose


load_dotenv()



## THEORY — Why Fairness Has MULTIPLE Incompatible Definitions


In [ ]:
# There is no single mathematical definition of "fair". Three families:
#
#   1. INDEPENDENCE (demographic parity):
#        P(decline | G = a) == P(decline | G = b)
#      Equal decision rates regardless of group.
#
#   2. SEPARATION (equalized odds):
#        TPR_a == TPR_b   AND   FPR_a == FPR_b
#      Equal error rates across groups.
#
#   3. SUFFICIENCY (calibration parity):
#        P(Y = 1 | score = p, G = a) == P(Y = 1 | score = p, G = b)
#      A score means the same default risk whatever your group.
#
# IMPOSSIBILITY (Chouldechova 2017; Kleinberg, Mullainathan & Raghavan
# 2016): when the groups' BASE RATES differ, these criteria are pairwise
# incompatible — no two of them can hold together, except for a perfect
# classifier or one that ignores the data. It is not "pick any two"; you
# choose which ONE to prioritise and document the trade-off.
#
# DISPARATE IMPACT RATIO = (approval rate of a group) / (approval rate of
# the most-approved group). The "four-fifths rule" (ratio < 0.8 = adverse
# impact) comes from the US EEOC Uniform Guidelines on Employee Selection
# Procedures (1978) — an employment-law screening heuristic that is widely
# borrowed as a rule of thumb in credit fairness audits.
#
# WHICH ATTRIBUTES: the dataset has race, gender and age. The model was
# trained WITH them as features (a choice many lenders avoid); the audit
# below measures outcomes per group either way — removing a column does not
# remove its proxies.



## TASK 2 — BUILD group-level audit machinery


Per-group rates for the three fairness families.


Approval-rate ratio of each group vs the most-approved group.


In [ ]:
bundle = build_shap_explainer()
model = bundle["model"]
X_test = bundle["X_test"]
y_test = bundle["y_test"]
y_pred = bundle["y_pred"]  # 1 = predicted default = DECLINE
y_proba = bundle["y_proba"]
feature_names: list[str] = bundle["feature_names"]
shap_vals = bundle["shap_vals"]
ordinal_mappings = bundle["ordinal_mappings"]


def group_report(
    y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray, groups: np.ndarray
) -> dict[str, dict[str, float]]:
    report: dict[str, dict[str, float]] = {}
    for g in sorted(set(groups.tolist())):
        m = groups == g
        pos, neg = m & (y_true == 1), m & (y_true == 0)
        declined = m & (y_pred == 1)
        report[g] = {
            "n": int(m.sum()),
            "base_rate": float(y_true[m].mean()),
            # TODO: approval rate = share of the group NOT declined (y_pred == 0)
            "approval_rate": ____,
            # TODO: TPR = share of actual defaulters (pos) that were declined;
            #       FPR = share of actual repayers (neg) that were declined
            "tpr": ____ if pos.any() else float("nan"),
            "fpr": ____ if neg.any() else float("nan"),
            "ppv": float(y_true[declined].mean()) if declined.any() else float("nan"),
            "mean_p": float(y_proba[m].mean()),
            # observed / predicted: 1.0 = calibrated "in the large" for this group
            "obs_over_pred": float(y_true[m].mean() / y_proba[m].mean()),
        }
    return report


def disparate_impact(report: dict[str, dict[str, float]]) -> dict[str, float]:
    best = max(r["approval_rate"] for r in report.values())
    # TODO: each group's approval rate divided by the best approval rate
    return ____



## TASK 3 — "TRAIN" = run the audit against the trained model


In [ ]:
reports: dict[str, dict[str, dict[str, float]]] = {}
di_by_attr: dict[str, dict[str, float]] = {}
group_labels: dict[str, np.ndarray] = {}
for attr in PROTECTED_ATTRIBUTES:
    # TODO: string group label per test row for this attribute
    # Hint: decode_group(X, feature_names, attribute, ordinal_mappings)
    groups = ____
    group_labels[attr] = groups
    reports[attr] = group_report(y_test, y_pred, y_proba, groups)
    di_by_attr[attr] = disparate_impact(reports[attr])

print_section("Fairness Audit — per-group rates (decline if P(default) >= 0.5)")
for attr in PROTECTED_ATTRIBUTES:
    print(f"\n  --- {attr} ---")
    print(
        f"  {'group':<10} {'n':>6} {'base':>6} {'approve':>8} {'DI':>6} "
        f"{'TPR':>6} {'FPR':>6} {'PPV':>6} {'obs/pred':>9}"
    )
    for g, r in reports[attr].items():
        di = di_by_attr[attr][g]
        flag = " <0.8" if di < 0.8 else ""
        print(
            f"  {g:<10} {r['n']:>6} {r['base_rate']:>6.3f} {r['approval_rate']:>8.3f} "
            f"{di:>6.3f} {r['tpr']:>6.3f} {r['fpr']:>6.3f} {r['ppv']:>6.3f} "
            f"{r['obs_over_pred']:>9.3f}{flag}"
        )

# Summary gaps per attribute (max - min across groups)
summary: dict[str, dict[str, float]] = {}
for attr, rep in reports.items():
    vals = list(rep.values())
    summary[attr] = {
        "min_di": min(di_by_attr[attr].values()),
        "tpr_gap": max(v["tpr"] for v in vals) - min(v["tpr"] for v in vals),
        "fpr_gap": max(v["fpr"] for v in vals) - min(v["fpr"] for v in vals),
        "calib_gap": max(v["obs_over_pred"] for v in vals) - min(v["obs_over_pred"] for v in vals),
        "base_rate_gap": max(v["base_rate"] for v in vals) - min(v["base_rate"] for v in vals),
    }



### Checkpoint


In [ ]:
assert set(reports) == set(PROTECTED_ATTRIBUTES), "Task 3: every protected attribute audited"
assert all(len(rep) >= 2 for rep in reports.values()), "Task 3: each attribute needs >= 2 groups"
assert all(
    sum(r["n"] for r in rep.values()) == len(y_test) for rep in reports.values()
), "Task 3: groups must partition the test set"
print("\n[ok] Checkpoint — per-group audit computed for race, gender and age\n")



## TASK 4 — VISUALISE: charts + the impossibility theorem on THIS model


In [ ]:
for attr in PROTECTED_ATTRIBUTES:
    rep = reports[attr]
    names = list(rep)
    fig = go.Figure()
    for metric, colour in [("approval_rate", "#10b981"), ("tpr", "#6366f1"), ("fpr", "#f43f5e")]:
        fig.add_trace(go.Bar(x=names, y=[rep[g][metric] for g in names], name=metric, marker_color=colour))
    fig.update_layout(
        title=f"Fairness by {attr}: approval rate, TPR, FPR",
        barmode="group",
        yaxis_title="Rate",
        height=420,
        legend=dict(orientation="h", y=-0.2),
    )
    path = OUTPUT_DIR / f"ex6_05_fairness_{attr}.html"
    fig.write_html(str(path))
    print(f"  Saved: {path}")

# Per-group reliability (calibration parity) for the attribute with the
# largest base-rate gap
focus_attr = max(summary, key=lambda a: summary[a]["base_rate_gap"])
fig_cal = go.Figure()
fig_cal.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect",
                             line=dict(dash="dash", color="#9ca3af")))
edges = np.linspace(0, 1, 11)
for g in reports[focus_attr]:
    m = group_labels[focus_attr] == g
    bin_idx = np.clip(np.digitize(y_proba[m], edges) - 1, 0, 9)
    xs, ys = [], []
    for b in range(10):
        sel = bin_idx == b
        if sel.sum() >= 30:
            xs.append(float(y_proba[m][sel].mean()))
            ys.append(float(y_test[m][sel].mean()))
    fig_cal.add_trace(go.Scatter(x=xs, y=ys, mode="lines+markers", name=g))
fig_cal.update_layout(
    title=f"Calibration parity by {focus_attr}: predicted vs observed default rate",
    xaxis_title="Mean predicted P(default) in bin",
    yaxis_title="Observed default rate",
    height=450,
)
cal_path = OUTPUT_DIR / f"ex6_05_calibration_parity_{focus_attr}.html"
fig_cal.write_html(str(cal_path))
print(f"  Saved: {cal_path}")



### Impossibility theorem, worked on this model


In [ ]:
# Chouldechova's identity links the three error rates in ANY group:
#
#     FPR = p / (1 - p) * (1 - PPV) / PPV * TPR        (p = base rate)
#
# Take the two groups of `focus_attr` with the most different base rates.
# Suppose the low-base-rate group had the SAME PPV (sufficiency-style
# parity) and the SAME TPR as the high-base-rate group. The identity then
# FIXES its FPR — and that FPR cannot equal the high group's FPR, so
# equalized odds must fail. Different base rates leave no way out.


def chouldechova_fpr(base_rate: float, ppv: float, tpr: float) -> float:
    # TODO: Chouldechova's identity FPR = p/(1-p) * (1-PPV)/PPV * TPR
    return ____


rep = reports[focus_attr]
hi_g = max(rep, key=lambda g: rep[g]["base_rate"])
lo_g = min(rep, key=lambda g: rep[g]["base_rate"])
hi, lo = rep[hi_g], rep[lo_g]
fpr_hi_check = chouldechova_fpr(hi["base_rate"], hi["ppv"], hi["tpr"])
# TODO: the FPR the low-base-rate group is forced to if it copies hi's PPV and TPR
fpr_lo_forced = ____

print_section(f"Impossibility theorem — {focus_attr}: {hi_g} vs {lo_g}", char="─")
print(f"  Base rates: {hi_g} = {hi['base_rate']:.3f}, {lo_g} = {lo['base_rate']:.3f}")
print(f"  {hi_g}: TPR={hi['tpr']:.3f}  PPV={hi['ppv']:.3f}  FPR={hi['fpr']:.3f}")
print(f"  Identity check for {hi_g}: FPR from formula = {fpr_hi_check:.3f}")
print(
    f"  If {lo_g} matched {hi_g}'s PPV and TPR, its FPR would be forced to "
    f"{fpr_lo_forced:.3f} — not {hi['fpr']:.3f}."
)
print("  -> With unequal base rates, equal PPV + equal TPR rules out equal FPR.")



### SHAP contribution of protected attributes


In [ ]:
importance_ranking = rank_features_by_mean_abs_shap(shap_vals, feature_names)
ranked_names = [n for n, _ in importance_ranking]
print_section("SHAP contribution of protected attributes", char="─")
for attr in PROTECTED_ATTRIBUTES:
    attr_shap = shap_vals[:, feature_index(feature_names, attr)]
    rank = ranked_names.index(attr) + 1
    action = "INVESTIGATE — in top 10" if rank <= 10 else "outside top 10"
    print(f"  {attr:<8} mean|SHAP|={np.abs(attr_shap).mean():.4f}  rank #{rank}/{len(feature_names)}  ({action})")



### Checkpoint


In [ ]:
assert abs(fpr_hi_check - hi["fpr"]) < 1e-9, "Chouldechova identity must hold exactly"
assert hi["base_rate"] > lo["base_rate"], "Worked example needs unequal base rates"
print("\n[ok] Checkpoint — impossibility identity verified on the model's own rates\n")

# INTERPRETATION: read the per-group table. Disparate impact flags groups
# whose approval rate is < 80% of the most-approved group. Large TPR/FPR
# gaps mean the model's errors land unevenly. obs/pred far from 1 means
# the probabilities are off for that group; compare the RATIO across
# groups — this model was trained with scale_pos_weight, so it
# over-predicts for everyone (see 5.5) and parity is about consistency.



## FAIRNESS AUDIT REPORT — every line computed above


In [ ]:
print_section("FAIRNESS AUDIT REPORT — Credit Default Model")
for attr in PROTECTED_ATTRIBUTES:
    s = summary[attr]
    worst = min(di_by_attr[attr], key=di_by_attr[attr].get)
    status = "FAILS four-fifths screen" if s["min_di"] < 0.8 else "passes four-fifths screen"
    print(
        f"  {attr:<7} min DI={s['min_di']:.3f} ({worst}; {status}) | "
        f"TPR gap={s['tpr_gap']:.3f} FPR gap={s['fpr_gap']:.3f} | "
        f"obs/pred gap={s['calib_gap']:.3f} | base-rate gap={s['base_rate_gap']:.3f}"
    )
failing = [a for a in PROTECTED_ATTRIBUTES if summary[a]["min_di"] < 0.8]
print(
    f"\n  Attributes needing escalation: {', '.join(failing) if failing else 'none'}"
    f"\n  Worked impossibility example used: {focus_attr} ({hi_g} vs {lo_g})"
)



## TASK 5 — APPLY: Fairness disclosure for an (illustrative) retail bank


In [ ]:
# SCENARIO (illustrative): a Singapore retail bank's model-risk committee
# asks for a fairness section in every credit model's documentation. The
# MAS FEAT principles (2018, non-binding) ask firms to assess AI-driven
# decisions for fairness and to justify the choices made; they do not
# prescribe a metric or a numeric threshold. A defensible disclosure:
#
#   1. Disparate impact per protected attribute (the table above)
#   2. TPR/FPR parity (equalized odds) per attribute
#   3. Calibration parity per attribute
#   4. WHICH criterion the bank prioritises, and why — the impossibility
#      example shows it cannot have all three when base rates differ
#   5. An escalation path when a screen fails
#
# What THIS audit shows: read the report lines above. Where an attribute
# fails the four-fifths screen together with a large base-rate gap (as age
# does if its line says so), the gap partly reflects genuinely different
# default rates — and the committee must decide whether the age effect
# is a legitimate risk factor or a proxy it should not use. That is a
# human decision; the pipeline's job is to make it visible.
#
# LIMITATION: a fairness audit identifies SYMPTOMS, not causes. Fixes —
# removing the attribute and its proxies, re-weighting, group-specific
# thresholds, a different model — each trade one criterion for another.



## DESTINATION-FIRST CLOSE — km.diagnose


In [ ]:
# This lesson built the fairness audit from primitives. kailash-ml packages
# the standard (non-group) diagnostic surface — per-class metrics,
# severity heuristics, confusion matrix — into one call; the group
# metrics above sit on top of it.

report = diagnose(model, kind="classical_classifier", data=(X_test, y_test), show=False)
print()
print("  km.diagnose model    : audited credit-default classifier")
print(f"  km.diagnose metrics  : {report.metrics}")
print(f"  km.diagnose severity : {report.severity}")



## REFLECTION


[x] Decoded race, gender and age bands from the encoded feature matrix
  [x] Measured disparate impact with the four-fifths screen
  [x] Measured TPR/FPR parity (equalized odds) and calibration parity
  [x] Verified Chouldechova's identity and used it to show, on this
      model's own base rates, why the criteria cannot all hold
  [x] Audited the SHAP contribution of protected attributes
  [x] Produced a computed fairness audit report

  KEY INSIGHT: Fairness is not a single number. It is a FAMILY of
  definitions that are mutually incompatible whenever base rates differ.
  The ML engineer's job is to measure all of them, show the trade-off,
  and make the choice legible to the people accountable for it.

  END OF EXERCISE 6. Next: Exercise 7 scales from a single model to a
  full Kailash Workflow with feature engineering, training, evaluation,
  and persistence.


In [ ]:
print_section("WHAT YOU'VE MASTERED")
print(
)

